# Rail Pulse — 01 Live Collector

This notebook continuously collects live Bluesky posts about Dutch passenger railways, runs the Hugging Face sentiment model locally, and stores results in SQLite.

It is designed around the Rail Pulse desk research:
- Dutch passenger railways;
- railway disruptions;
- NS / ProRail context;
- SDG 9;
- a railway communications or service-quality analyst as the user.

Keep this notebook kernel and your computer awake during collection.

**Important:** `events_seen` is the number of Bluesky events examined. Only rows actually stored in the `posts` table count toward the 300-item requirement.

## 1. Install packages
Run this once in your `.venv`.

In [3]:
%pip install -q -U torch transformers websocket-client sentencepiece protobuf tiktoken pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Imports, database and configuration

In [4]:
from pathlib import Path
import sqlite3
import json
import re
import hashlib
import random
import threading
import time
from datetime import datetime, timezone

from websocket import create_connection, WebSocketTimeoutException
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DB = (DATA_DIR / "rail_pulse.sqlite3").resolve()

PRIMARY_MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

JETSTREAM_BASE = (
    "wss://jetstream2.us-east.bsky.network/subscribe"
    "?wantedCollections=app.bsky.feed.post"
)

print("Database:", DB)
print("Primary model:", PRIMARY_MODEL)

Database: C:\Users\ikben\Desktop\test\a\data\rail_pulse.sqlite3
Primary model: cardiffnlp/twitter-xlm-roberta-base-sentiment


In [5]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def connect():
    db = sqlite3.connect(
        str(DB),
        timeout=30,
        check_same_thread=False,
    )

    db.execute("PRAGMA journal_mode=WAL")
    db.execute("PRAGMA busy_timeout=30000")

    db.execute("""
        CREATE TABLE IF NOT EXISTS posts (
            id TEXT PRIMARY KEY,
            text_hash TEXT NOT NULL UNIQUE,
            source TEXT NOT NULL,
            created_at TEXT NOT NULL,
            collected_at TEXT NOT NULL,
            text TEXT NOT NULL,
            language_hint TEXT,
            category TEXT NOT NULL,
            match_reason TEXT NOT NULL,
            matched_terms TEXT NOT NULL,
            disruption INTEGER NOT NULL,
            sentiment TEXT NOT NULL,
            confidence REAL NOT NULL,
            model TEXT NOT NULL
        )
    """)

    db.execute("""
        CREATE TABLE IF NOT EXISTS heartbeat (
            id INTEGER PRIMARY KEY CHECK(id=1),
            last_seen TEXT,
            events_seen INTEGER NOT NULL DEFAULT 0,
            stored_this_session INTEGER NOT NULL DEFAULT 0,
            reconnects INTEGER NOT NULL DEFAULT 0,
            last_error TEXT,
            last_time_us INTEGER
        )
    """)

    db.execute("""
        CREATE TABLE IF NOT EXISTS sessions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            started_at TEXT NOT NULL,
            last_seen TEXT NOT NULL,
            stopped_at TEXT
        )
    """)

    db.execute("""
        CREATE TABLE IF NOT EXISTS meta (
            key TEXT PRIMARY KEY,
            value TEXT
        )
    """)

    db.commit()
    return db


def get_meta(key, default=None):
    with connect() as db:
        row = db.execute(
            "SELECT value FROM meta WHERE key=?",
            (key,),
        ).fetchone()

    return default if row is None else row[0]


def set_meta(key, value):
    with connect() as db:
        db.execute(
            """
            INSERT INTO meta(key, value)
            VALUES (?, ?)
            ON CONFLICT(key) DO UPDATE SET value=excluded.value
            """,
            (key, str(value)),
        )
        db.commit()


connect().close()
print("Database schema ready.")

Database schema ready.


## 3. Dutch railway relevance filter

This fixes the false positives from the earlier versions.

It does **not** collect generic `metro`, `tram` or `bus` posts.

Strong Dutch railway terms can pass directly. Ambiguous or international railway terms such as `InterCity`, `station`, `sprinter`, `perron`, `conducteur`, `spoor` and `NS` require Dutch-language metadata or explicit Netherlands railway context.

Every stored post also gets a separate disruption flag.

### Important fix in this version

The previous notebook accidentally used an over-escaped regular-expression boundary, so short terms such as `ns` could match *inside* unrelated words. This version uses real word boundaries (`\\w`) and includes the actual false-positive examples from the test run as self-tests.


In [6]:
STRONG_DUTCH_RAIL_TERMS = [
    "trein",
    "treinen",
    "treinreis",
    "treinreizen",
    "treinrit",
    "treinritten",
    "treinkaartje",
    "treinkaartjes",
    "treinreiziger",
    "treinreizigers",
    "treinverkeer",
    "treinvertraging",
    "treinstoring",
    "treinuitval",
    "spoorstoring",
    "spoorwerkzaamheden",
    "wisselstoring",
    "seinstoring",
    "bovenleiding",
    "prorail",
    "nederlandse spoorwegen",
    "ns trein",
    "ns treinen",
    "ns vertraging",
    "ns storing",
    "ns reisplanner",
    "ns app",
    "ns flex",
]

AMBIGUOUS_RAIL_TERMS = [
    "intercity",
    "sprinter",
    "station",
    "stations",
    "spoor",
    "spoorweg",
    "spoorwegen",
    "spoorlijn",
    "perron",
    "perrons",
    "conducteur",
    "conducteurs",
    "ns",
    "icng",
]

EXPLICIT_NL_RAIL_CONTEXT = [
    "prorail",
    "nederlandse spoorwegen",
    "ns trein",
    "ns treinen",
    "ns intercity",
    "ns sprinter",
    "ns vertraging",
    "ns storing",
    "ns reisplanner",
    "ns app",
    "ns flex",
    "utrecht centraal",
    "amsterdam centraal",
    "rotterdam centraal",
    "den haag centraal",
    "leiden centraal",
    "eindhoven centraal",
    "arnhem centraal",
    "nijmegen centraal",
    "breda centraal",
    "amersfoort centraal",
    "station schiphol",
    "schiphol station",
    "station zwolle",
    "zwolle station",
    "station groningen",
    "groningen station",
    "station delft",
    "delft station",
    "station haarlem",
    "haarlem station",
    "station dordrecht",
    "dordrecht station",
    "station schiedam",
    "schiedam station",
]

DUTCH_LOCATION_CONTEXT = [
    "nederland",
    "nederlandse",
    "schiphol",
    "amsterdam",
    "rotterdam",
    "den haag",
    "utrecht",
    "eindhoven",
    "groningen",
    "maastricht",
    "leiden",
    "delft",
    "haarlem",
    "zwolle",
    "arnhem",
    "nijmegen",
    "breda",
    "tilburg",
    "almere",
    "amersfoort",
    "dordrecht",
    "schiedam",
    "vlaardingen",
]

DISRUPTION_TERMS = [
    "vertraging",
    "vertraagd",
    "storing",
    "uitval",
    "uitgevallen",
    "valt uit",
    "rijdt niet",
    "rijden niet",
    "geen treinen",
    "minder treinen",
    "geannuleerd",
    "afgelast",
    "gestrand",
    "werkzaamheden",
    "hinder",
    "kapot",
    "defect",
    "aansluiting gemist",
    "aansluiting missen",
    "te laat",
    "overvol",
    "drukte",
    "niet bereikbaar",
    "wisselprobleem",
    "seinprobleem",
    "bovenleiding",
    "delayed",
    "cancelled",
    "canceled",
    "disruption",
]


def term_pattern(term):
    return r"(?<!\w)" + re.escape(term.casefold()) + r"(?!\w)"


def find_terms(text, terms):
    lowered = text.casefold()
    found = []

    for term in terms:
        if re.search(term_pattern(term), lowered):
            found.append(term)

    return found


def has_dutch_language(langs):
    if not isinstance(langs, list):
        return False

    return any(
        isinstance(lang, str)
        and lang.casefold().startswith("nl")
        for lang in langs
    )


def has_explicit_non_dutch_language(langs):
    if not isinstance(langs, list) or not langs:
        return False

    return not has_dutch_language(langs)


def classify_relevance(text, langs=None):
    strong = find_terms(text, STRONG_DUTCH_RAIL_TERMS)
    explicit_context = find_terms(text, EXPLICIT_NL_RAIL_CONTEXT)
    locations = find_terms(text, DUTCH_LOCATION_CONTEXT)

    if strong:
        # If Bluesky explicitly marks the post as non-Dutch,
        # require extra Netherlands-specific context to avoid
        # foreign false positives such as Afrikaans "trein".
        if (
            has_explicit_non_dutch_language(langs)
            and not explicit_context
            and not locations
            and "prorail" not in strong
            and "nederlandse spoorwegen" not in strong
        ):
            return None

        return {
            "category": "rail",
            "reason": "strong_dutch_rail_term",
            "terms": sorted(
                set(strong + explicit_context + locations)
            ),
        }

    ambiguous = find_terms(text, AMBIGUOUS_RAIL_TERMS)

    if ambiguous:
        if has_dutch_language(langs):
            return {
                "category": "rail",
                "reason": "ambiguous_term_with_dutch_language",
                "terms": sorted(
                    set(ambiguous + explicit_context + locations)
                ),
            }

        if explicit_context:
            return {
                "category": "rail",
                "reason": "ambiguous_term_with_explicit_nl_context",
                "terms": sorted(
                    set(ambiguous + explicit_context + locations)
                ),
            }

        if locations:
            return {
                "category": "rail",
                "reason": "rail_term_with_nl_location",
                "terms": sorted(
                    set(ambiguous + locations)
                ),
            }

    return None


def disruption_flag(text):
    return int(bool(find_terms(text, DISRUPTION_TERMS)))


filter_tests = [
    # Known false positives from earlier runs — all must be rejected
    ("🚫 Cancelled SEPTA Metro trips today", ["en"], False),
    ("[19:40] TOKYO METRO Yurakucho Line delayed", ["en"], False),
    ("METRO2039 preorder announced", ["pl"], False),
    ("Dati del 14/10/2026 Treni Regolari: 9138 InterCity: 9:30 min.", ["it"], False),
    ("Wel geinig, dat zo'n LePen niet meer wil betalen aan de Europese Unie dan kleine landen als België of Nederland.", ["nl"], False),
    ("Try to eat really well: nutrition expert says healthy eating is easier than you think.", ["nl"], False),
    ("What's Dionysus Got to Do With Me? Rotterdam", ["en"], False),
    ("Poor Cherries #girlshavingfun #movie #metrostation", ["en"], False),

    # Genuine Dutch-rail examples — all must be accepted
    ("Mijn trein naar Utrecht heeft weer vertraging", ["nl"], True),
    ("Treinstoring tussen Leiden en Den Haag", [], True),
    ("Intercity naar Rotterdam rijdt niet", ["nl"], True),
    ("Utrecht Centraal is vandaag erg druk", ["nl"], True),
    ("ProRail meldt werkzaamheden aan het spoor", ["nl"], True),
]

print("Filter self-test:")
for text, langs, expected in filter_tests:
    actual = classify_relevance(text, langs) is not None

    print(
        actual == expected,
        "| expected:", expected,
        "| got:", actual,
        "|", text
    )

Filter self-test:
True | expected: False | got: False | 🚫 Cancelled SEPTA Metro trips today
True | expected: False | got: False | [19:40] TOKYO METRO Yurakucho Line delayed
True | expected: False | got: False | METRO2039 preorder announced
True | expected: False | got: False | Dati del 14/10/2026 Treni Regolari: 9138 InterCity: 9:30 min.
True | expected: False | got: False | Wel geinig, dat zo'n LePen niet meer wil betalen aan de Europese Unie dan kleine landen als België of Nederland.
True | expected: False | got: False | Try to eat really well: nutrition expert says healthy eating is easier than you think.
True | expected: False | got: False | What's Dionysus Got to Do With Me? Rotterdam
True | expected: False | got: False | Poor Cherries #girlshavingfun #movie #metrostation
True | expected: True | got: True | Mijn trein naar Utrecht heeft weer vertraging
True | expected: True | got: True | Treinstoring tussen Leiden en Den Haag
True | expected: True | got: True | Intercity naar Rott

## 4. Load the Hugging Face sentiment transformer

`use_fast=False` is intentional. It uses the SentencePiece tokenizer directly and avoids the tokenizer-conversion issue you encountered earlier.

In [7]:
def preprocess_for_model(text):
    cleaned = []

    for token in text.split():
        if token.startswith("@") and len(token) > 1:
            cleaned.append("@user")
        elif token.startswith("http"):
            cleaned.append("http")
        else:
            cleaned.append(token)

    return " ".join(cleaned)


def normalize_primary_label(label):
    value = str(label).strip().lower()

    return {
        "label_0": "negative",
        "label_1": "neutral",
        "label_2": "positive",
    }.get(value, value)


print("Loading Hugging Face model. First run may take a while...")

primary_tokenizer = AutoTokenizer.from_pretrained(
    PRIMARY_MODEL,
    use_fast=False,
)

primary_model = AutoModelForSequenceClassification.from_pretrained(
    PRIMARY_MODEL
)

primary_pipe = pipeline(
    "text-classification",
    model=primary_model,
    tokenizer=primary_tokenizer,
    device=-1,
)

print("Model ready.")

Loading Hugging Face model. First run may take a while...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model ready.


## 5. Optional reset

Because you are starting from zero, you normally do not need this.

If you first run a short quality test and then want to erase the test rows before beginning the official 48-hour collection, stop the collector and run:

`reset_official_run(confirm=True)`

Never reset after the official collection has started.

In [8]:
def reset_official_run(confirm=False):
    if confirm is not True:
        print(
            "Nothing deleted. "
            "Use reset_official_run(confirm=True) only when you really "
            "want to erase ALL collected Rail Pulse data."
        )
        return

    thread = globals().get("collector_thread")

    if thread is not None and thread.is_alive():
        raise RuntimeError(
            "Collector is still running. "
            "Run stop_collector(), wait until thread_alive is False, "
            "then reset."
        )

    with connect() as db:
        db.execute("DELETE FROM posts")
        db.execute("DELETE FROM heartbeat")
        db.execute("DELETE FROM sessions")
        db.execute("DELETE FROM meta")
        db.commit()

    print("All collected test data deleted.")

## 6. Continuous collector

This defines the background collector, reconnect logic, cursor resume, deduplication, heartbeat and session tracking.

In [9]:
stop_event = threading.Event()
collector_thread = None

status = {
    "events": 0,
    "stored": 0,
    "reconnects": 0,
    "last_error": None,
    "last_time_us": None,
}


def load_saved_cursor():
    with connect() as db:
        row = db.execute(
            "SELECT last_time_us FROM heartbeat WHERE id=1"
        ).fetchone()

    if row and row[0]:
        return int(row[0])

    return None


def build_jetstream_url(cursor=None):
    if cursor is None:
        return JETSTREAM_BASE

    # Replay three seconds after reconnect.
    # Duplicate IDs/text are ignored in SQLite.
    replay_cursor = max(
        0,
        int(cursor) - 3_000_000
    )

    return f"{JETSTREAM_BASE}&cursor={replay_cursor}"


def write_heartbeat(db, session_id):
    stamp = utc_now()

    db.execute(
        """
        INSERT INTO heartbeat(
            id,
            last_seen,
            events_seen,
            stored_this_session,
            reconnects,
            last_error,
            last_time_us
        )
        VALUES(1, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(id) DO UPDATE SET
            last_seen=excluded.last_seen,
            events_seen=excluded.events_seen,
            stored_this_session=excluded.stored_this_session,
            reconnects=excluded.reconnects,
            last_error=excluded.last_error,
            last_time_us=excluded.last_time_us
        """,
        (
            stamp,
            status["events"],
            status["stored"],
            status["reconnects"],
            status["last_error"],
            status["last_time_us"],
        ),
    )

    db.execute(
        "UPDATE sessions SET last_seen=? WHERE id=?",
        (
            stamp,
            session_id,
        ),
    )

    db.commit()


def collect_forever():
    db = connect()

    official_start = get_meta("official_start")

    if official_start is None:
        official_start = utc_now()
        set_meta(
            "official_start",
            official_start
        )

    started = utc_now()

    session_id = db.execute(
        """
        INSERT INTO sessions(
            started_at,
            last_seen
        )
        VALUES (?, ?)
        """,
        (
            started,
            started,
        ),
    ).lastrowid

    db.commit()

    saved_cursor = load_saved_cursor()
    status["last_time_us"] = saved_cursor

    backoff = 2
    last_heartbeat_write = 0.0

    try:
        while not stop_event.is_set():
            ws = None

            try:
                url = build_jetstream_url(
                    status["last_time_us"]
                )

                ws = create_connection(
                    url,
                    timeout=30,
                )

                ws.settimeout(15)

                backoff = 2
                status["last_error"] = None

                print(
                    "Connected:",
                    utc_now(),
                    "| cursor:",
                    status["last_time_us"],
                )

                while not stop_event.is_set():
                    now_mono = time.monotonic()

                    if (
                        now_mono
                        - last_heartbeat_write
                        >= 5
                    ):
                        write_heartbeat(
                            db,
                            session_id
                        )

                        last_heartbeat_write = (
                            now_mono
                        )

                    try:
                        raw = ws.recv()

                    except WebSocketTimeoutException:
                        continue

                    if not raw:
                        raise ConnectionError(
                            "Jetstream socket closed"
                        )

                    try:
                        event = json.loads(raw)

                    except (
                        ValueError,
                        TypeError,
                    ):
                        continue

                    event_time_us = event.get(
                        "time_us"
                    )

                    if isinstance(
                        event_time_us,
                        int
                    ):
                        status["last_time_us"] = (
                            event_time_us
                        )

                    status["events"] += 1

                    if event.get("kind") != "commit":
                        continue

                    commit = event.get("commit") or {}

                    if (
                        commit.get("collection")
                        != "app.bsky.feed.post"
                    ):
                        continue

                    if (
                        commit.get("operation")
                        != "create"
                    ):
                        continue

                    record = commit.get("record") or {}

                    text = record.get("text", "")
                    langs = record.get("langs") or []

                    if not isinstance(text, str):
                        continue

                    text = text.strip()

                    if not text:
                        continue

                    relevance = classify_relevance(
                        text,
                        langs,
                    )

                    if relevance is None:
                        continue

                    did = event.get("did")
                    rkey = commit.get("rkey")

                    if not did or not rkey:
                        continue

                    # DID/URI is used only in memory to make a one-way hash.
                    uri = (
                        f"at://{did}/"
                        f"app.bsky.feed.post/"
                        f"{rkey}"
                    )

                    item_id = hashlib.sha256(
                        uri.encode("utf-8")
                    ).hexdigest()

                    normalized_text = re.sub(
                        r"\s+",
                        " ",
                        text.casefold(),
                    ).strip()

                    text_hash = hashlib.sha256(
                        normalized_text.encode(
                            "utf-8"
                        )
                    ).hexdigest()

                    already_exists = db.execute(
                        """
                        SELECT 1
                        FROM posts
                        WHERE id=?
                           OR text_hash=?
                        """,
                        (
                            item_id,
                            text_hash,
                        ),
                    ).fetchone()

                    if already_exists:
                        continue

                    created_at = (
                        record.get("createdAt")
                        or utc_now()
                    )

                    try:
                        created_dt = datetime.fromisoformat(
                            created_at.replace(
                                "Z",
                                "+00:00"
                            )
                        )

                        if created_dt.tzinfo is None:
                            created_dt = created_dt.replace(
                                tzinfo=timezone.utc
                            )

                        created_at = (
                            created_dt
                            .astimezone(
                                timezone.utc
                            )
                            .isoformat()
                        )

                    except (
                        ValueError,
                        TypeError,
                    ):
                        created_at = utc_now()

                    model_text = preprocess_for_model(
                        text
                    )

                    try:
                        prediction = primary_pipe(
                            model_text,
                            truncation=True,
                            max_length=512,
                        )[0]

                    except Exception as exc:
                        status["last_error"] = (
                            "Model error: "
                            + str(exc)
                        )

                        print(
                            status["last_error"]
                        )

                        continue

                    sentiment = normalize_primary_label(
                        prediction["label"]
                    )

                    confidence = float(
                        prediction["score"]
                    )

                    is_disruption = disruption_flag(
                        text
                    )

                    if isinstance(langs, list):
                        language_hint = ",".join(
                            str(x)
                            for x in langs
                        )
                    else:
                        language_hint = ""

                    matched_terms = ",".join(
                        relevance["terms"]
                    )

                    try:
                        db.execute(
                            """
                            INSERT INTO posts(
                                id,
                                text_hash,
                                source,
                                created_at,
                                collected_at,
                                text,
                                language_hint,
                                category,
                                match_reason,
                                matched_terms,
                                disruption,
                                sentiment,
                                confidence,
                                model
                            )
                            VALUES (
                                ?, ?, ?, ?, ?, ?, ?,
                                ?, ?, ?, ?, ?, ?, ?
                            )
                            """,
                            (
                                item_id,
                                text_hash,
                                "bluesky_jetstream",
                                created_at,
                                utc_now(),
                                text,
                                language_hint,
                                relevance["category"],
                                relevance["reason"],
                                matched_terms,
                                is_disruption,
                                sentiment,
                                confidence,
                                PRIMARY_MODEL,
                            ),
                        )

                        db.commit()

                    except sqlite3.IntegrityError:
                        continue

                    status["stored"] += 1

                    print(
                        f"Stored #{status['stored']} | "
                        f"{sentiment} "
                        f"{confidence:.3f} | "
                        f"disruption={is_disruption} | "
                        f"{text[:140]}"
                    )

            except Exception as exc:
                status["reconnects"] += 1
                status["last_error"] = str(exc)

                print(
                    "Reconnect after error:",
                    repr(exc),
                )

                try:
                    write_heartbeat(
                        db,
                        session_id
                    )
                except Exception:
                    pass

                stop_event.wait(
                    backoff
                    + random.random()
                )

                backoff = min(
                    backoff * 2,
                    120,
                )

            finally:
                if ws is not None:
                    try:
                        ws.close()
                    except Exception:
                        pass

    finally:
        try:
            write_heartbeat(
                db,
                session_id
            )

            stopped = utc_now()

            db.execute(
                """
                UPDATE sessions
                SET stopped_at=?,
                    last_seen=?
                WHERE id=?
                """,
                (
                    stopped,
                    stopped,
                    session_id,
                ),
            )

            db.commit()

        finally:
            db.close()


def start_collector():
    global collector_thread

    if (
        collector_thread is not None
        and collector_thread.is_alive()
    ):
        print(
            "Collector is already running."
        )
        return

    stop_event.clear()

    collector_thread = threading.Thread(
        target=collect_forever,
        daemon=True,
        name="rail-pulse-collector",
    )

    collector_thread.start()

    print(
        "Collector started. "
        "Keep this kernel and computer awake."
    )


def stop_collector():
    stop_event.set()

    if collector_thread is not None:
        collector_thread.join(
            timeout=20
        )

    print(
        "Stop requested. "
        "Use get_status() to confirm "
        "thread_alive becomes False."
    )

## 7. Start the collector
Run this once.

In [10]:
start_collector()

Collector started. Keep this kernel and computer awake.


## 8. Status check

A healthy collector should normally show:
- `thread_alive: True`
- `last_error: None`
- `events` increasing over time

The stored count can rise slowly because only railway-relevant posts are stored.

In [11]:
def get_status():
    with connect() as db:
        total = db.execute(
            "SELECT COUNT(*) FROM posts"
        ).fetchone()[0]

        disruptions = db.execute(
            """
            SELECT COUNT(*)
            FROM posts
            WHERE disruption=1
            """
        ).fetchone()[0]

        heartbeat = db.execute(
            """
            SELECT
                last_seen,
                events_seen,
                stored_this_session,
                reconnects,
                last_error,
                last_time_us
            FROM heartbeat
            WHERE id=1
            """
        ).fetchone()

    official_start = get_meta(
        "official_start"
    )

    elapsed_hours = None
    stored_per_hour = None
    projected_48h = None

    if official_start:
        start_dt = datetime.fromisoformat(
            official_start.replace(
                "Z",
                "+00:00"
            )
        )

        if start_dt.tzinfo is None:
            start_dt = start_dt.replace(
                tzinfo=timezone.utc
            )

        elapsed_hours = max(
            0.0,
            (
                datetime.now(
                    timezone.utc
                )
                - start_dt.astimezone(
                    timezone.utc
                )
            ).total_seconds()
            / 3600,
        )

        if elapsed_hours >= 1 and total >= 10:
            stored_per_hour = (
                total / elapsed_hours
            )

            projected_48h = (
                stored_per_hour * 48
            )

    return {
        "all_stored": total,
        "rail": total,
        "disruption_flagged": disruptions,
        "target_300_met": total >= 300,
        "official_start": official_start,
        "elapsed_hours": (
            None
            if elapsed_hours is None
            else round(
                elapsed_hours,
                2
            )
        ),
        "stored_per_hour": (
            None
            if stored_per_hour is None
            else round(
                stored_per_hour,
                2
            )
        ),
        "projected_48h_items": (
            None
            if projected_48h is None
            else round(
                projected_48h
            )
        ),
        "thread_alive": bool(
            collector_thread
            and collector_thread.is_alive()
        ),
        "session_status": dict(
            status
        ),
        "heartbeat": heartbeat,
    }


get_status()

{'all_stored': 72,
 'rail': 72,
 'disruption_flagged': 13,
 'target_300_met': False,
 'official_start': '2026-10-14T22:41:12.211882+00:00',
 'elapsed_hours': 22.3,
 'stored_per_hour': 3.23,
 'projected_48h_items': 155,
 'thread_alive': True,
 'session_status': {'events': 0,
  'stored': 0,
  'reconnects': 0,
  'last_error': None,
  'last_time_us': 1792061202773544},
 'heartbeat': ('2026-10-15T10:46:43.318815+00:00',
  7705,
  0,
  0,
  None,
  1792061202773544)}

## 9. Inspect newest stored posts

Run this regularly during the first hour. The count alone is not enough: manually check that stored posts are genuinely about Dutch passenger railways.

In [12]:
def show_latest(limit=20):
    with connect() as db:
        rows = db.execute(
            """
            SELECT
                created_at,
                language_hint,
                match_reason,
                matched_terms,
                disruption,
                sentiment,
                confidence,
                text
            FROM posts
            ORDER BY collected_at DESC
            LIMIT ?
            """,
            (
                int(limit),
            ),
        ).fetchall()

    if not rows:
        print(
            "No posts stored yet."
        )
        return

    for row in rows:
        (
            created_at,
            language_hint,
            match_reason,
            matched_terms,
            disruption,
            sentiment,
            confidence,
            text,
        ) = row

        print()
        print(
            "TIME:",
            created_at
        )
        print(
            "LANG:",
            language_hint
        )
        print(
            "MATCH:",
            match_reason,
            "|",
            matched_terms
        )
        print(
            "DISRUPTION:",
            disruption
        )
        print(
            "SENTIMENT:",
            sentiment,
            "| CONFIDENCE:",
            round(
                confidence,
                3
            )
        )
        print(
            "TEXT:",
            text
        )
        print(
            "-" * 90
        )


show_latest(20)


TIME: 2026-10-15T10:32:53.781000+00:00
LANG: nl
MATCH: strong_dutch_rail_term | trein
DISRUPTION: 0
SENTIMENT: positive | CONFIDENCE: 0.604
TEXT: Deze trein is een stuk rustiger dan de vorige. Het is ook begonnen met regenen.
------------------------------------------------------------------------------------------

TIME: 2026-10-15T10:31:12.033759+00:00
LANG: nl
MATCH: strong_dutch_rail_term | dordrecht,rotterdam,trein
DISRUPTION: 1
SENTIMENT: negative | CONFIDENCE: 0.575
TEXT: STORING: defecte trein tussen Rotterdam - Dordrecht.

Dit duurt tot ongeveer 13:00 uur.
------------------------------------------------------------------------------------------

TIME: 2026-10-15T10:17:08.695000+00:00
LANG: nl
MATCH: ambiguous_term_with_dutch_language | station
DISRUPTION: 0
SENTIMENT: positive | CONFIDENCE: 0.48
TEXT: Onder de kap bij station Aurillac - met nog twee minuten marge tot vertrek - staat een nachttrein die al een beste tijd geleden vanuit Parijs hier is aangekomen. Ik ga verder m

## 10. Stop only when intentionally ending collection

Run:

```python
stop_collector()
```

Do not run this during the official 48-hour period unless necessary.

In [ ]:
get_status()

{'all_stored': 72,
 'rail': 72,
 'disruption_flagged': 13,
 'target_300_met': False,
 'official_start': '2026-10-14T22:41:12.211882+00:00',
 'elapsed_hours': 22.3,
 'stored_per_hour': 3.23,
 'projected_48h_items': 155,
 'thread_alive': True,
 'session_status': {'events': 0,
  'stored': 0,
  'reconnects': 0,
  'last_error': None,
  'last_time_us': 1792061202773544},
 'heartbeat': ('2026-10-15T10:46:43.318815+00:00',
  7705,
  0,
  0,
  None,
  1792061202773544)}

Connected: 2026-10-15T20:58:59.314566+00:00 | cursor: 1792061202773544
Stored #1 | negative 0.871 | disruption=1 | En nu defect spoor waar ik heen wilde. Nou laat maar dan 😂


In [ ]:
stop_collector()


TIME: 2026-10-15T10:53:46.914000+00:00
LANG: nl
MATCH: ambiguous_term_with_dutch_language | spoor
DISRUPTION: 1
SENTIMENT: negative | CONFIDENCE: 0.871
TEXT: En nu defect spoor waar ik heen wilde. Nou laat maar dan 😂
------------------------------------------------------------------------------------------

TIME: 2026-10-15T10:32:53.781000+00:00
LANG: nl
MATCH: strong_dutch_rail_term | trein
DISRUPTION: 0
SENTIMENT: positive | CONFIDENCE: 0.604
TEXT: Deze trein is een stuk rustiger dan de vorige. Het is ook begonnen met regenen.
------------------------------------------------------------------------------------------

TIME: 2026-10-15T10:31:12.033759+00:00
LANG: nl
MATCH: strong_dutch_rail_term | dordrecht,rotterdam,trein
DISRUPTION: 1
SENTIMENT: negative | CONFIDENCE: 0.575
TEXT: STORING: defecte trein tussen Rotterdam - Dordrecht.

Dit duurt tot ongeveer 13:00 uur.
------------------------------------------------------------------------------------------

TIME: 2026-10-15T10:17:08.

Stored #2 | positive 0.647 | disruption=0 | 15aug 2020 kreeg ik de sleutels voor The wildknittingroom 
15oktober 2026 leverde ik de ruimte niet veegschoon, maar stofzuigschoon op.
“Lea
Stored #3 | negative 0.621 | disruption=1 | Storing voorbij: defecte trein Rotterdam - Dordrecht
Stored #4 | negative 0.610 | disruption=1 | STORING: defecte trein tussen Sittard - Heerlen.

Dit duurt tot ongeveer 14:45 uur.
Stored #5 | neutral 0.504 | disruption=0 | Geen treinverkeer tussen Wondelgem en Eeklo na ongeval in Sleidinge
Stored #6 | negative 0.606 | disruption=1 | Geen treinverkeer tussen Wondelgem en Eeklo na ongeval in Sleidinge

Er rijden momenteel geen treinen tussen Wondelgem (Gent) en Eeklo door 
Stored #7 | neutral 0.525 | disruption=0 | Ik had hem al zien liggen en nu is hij eindelijk open: de Nieuwe Yalurivierbrug. De oude is zo smal dat er maar in een richting tegelijk ver
Stored #8 | neutral 0.847 | disruption=0 | Smaakt naar meer: Mirte fietst van Praag terug naar huis

Mirte Weg